# 8-5、8-6 Cloud Function 自動交易

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finlab-python/hahow-crypto-course/blob/main/8-5_cloud_function.ipynb)

**對應影片**：第 8 章 單元 8-5【雲端自動化】用 Google Cloud Function 建構自動交易介面、8-6【雲端自動化】用 Cloud Scheduler 做排程

**和影片的差異**

- 部署用的程式與完整的部署、排程指令放在 [`cloud_function/`](https://github.com/finlab-python/hahow-crypto-course/tree/main/cloud_function) 資料夾（`main.py`、`requirements.txt`、`README.md`）。
- 這本筆記本在部署前先於本機 / Colab 用 dry run 執行同一份 `main.py`，確認報表內容。
- 改用 `gcloud` 指令部署第 2 代 Cloud Functions（Python 3.12，區域 asia-east1 台灣），金鑰放 Secret Manager，函式不公開。
- VETBTC 已於 2026 年 3 月下架，altcoin 策略改用 BNBBTC。

In [1]:
%pip install -q "finlab_crypto[trade]==0.3.0" functions-framework

Note: you may need to restart the kernel to use updated packages.


# 取得部署用的程式

在 Colab 上會從 GitHub 下載 `cloud_function/main.py` 與 `requirements.txt`；本機執行時直接使用資料夾裡的檔案。

In [2]:
from pathlib import Path
from urllib.request import urlretrieve

SOURCE = 'https://raw.githubusercontent.com/finlab-python/hahow-crypto-course/main/cloud_function/'
folder = Path('cloud_function')
folder.mkdir(exist_ok=True)
for name in ['main.py', 'requirements.txt']:
    if not (folder / name).exists():
        urlretrieve(SOURCE + name, folder / name)

print((folder / 'main.py').read_text())

"""Cloud Function: rebalance a Binance spot account with the course strategies.

HTTP entry point ``main``: ``?mode=TEST|LIMIT|MARKET`` or JSON body ``{"mode": ...}``.
API keys come from the environment (Secret Manager), never from the source.
"""
import os

import functions_framework

from finlab_crypto.indicators import trends
from finlab_crypto.online import TradingMethod, TradingPortfolio, render_html
from finlab_crypto.strategy import Strategy

DEFAULT_MODE = 'TEST'
MARGIN_USDT = 1000


@Strategy(name='sma', n1=20, n2=40)
def trend_strategy(ohlcv):
    fast = trends[trend_strategy.name](ohlcv.close, trend_strategy.n1)
    slow = trends[trend_strategy.name](ohlcv.close, trend_strategy.n2)
    entries = (fast > slow) & (fast.shift() < slow.shift())
    exits = (fast < slow) & (fast.shift() > slow.shift())
    return entries, exits, {'overlaps': {'trend1': fast, 'trend2': slow}}


TRADING_METHODS = [
    TradingMethod(
        symbols=['XRPBTC', 'ADABTC', 'LINKBTC', 'ETHBTC', 'BNBBTC

# 8-5 在部署前試跑

模擬一次 HTTP 請求 `?mode=TEST`。沒有設定 `BINANCE_KEY` 時是 dry run：訂單只列出、不送出。

**重要**：下單使用 `api.binance.com`，它會拒絕美國 IP（HTTP 451）。Colab 主機在美國，所以 Colab 上只能 dry run（計算訊號與訂單、不送出）。真正下單請在自己的電腦（台灣）執行，或部署到 asia-east1（台灣）的 Cloud Function（見 8-5）。

In [3]:
import sys

import flask
from IPython.display import HTML

sys.path.insert(0, str(folder))
import main

with flask.Flask(__name__).test_request_context('/?mode=TEST'):
    report = main.main(flask.request)

HTML(report)

|---------EXECUTION LOG----------|
| time: 2026-09-24 11:40:14 (dry run, nothing sent)
| TEST ADAUSDT BUY 3623.1 dry run
| TEST BTCUSDT BUY 0.01 dry run
| TEST ETHUSDT BUY 0.3172 dry run
| TEST LINKUSDT BUY 64.26 dry run
| TEST XRPUSDT BUY 544.3 dry run


# 部署到 Google Cloud Function、用 Cloud Scheduler 排程

照 [`cloud_function/README.md`](https://github.com/finlab-python/hahow-crypto-course/tree/main/cloud_function) 的步驟，在自己的電腦或
[Cloud Shell](https://shell.cloud.google.com/) 執行 `gcloud` 指令：

1. 把幣安 API 金鑰存進 Secret Manager
2. `gcloud functions deploy`：部署到 asia-east1，不公開
3. 用 `curl` 帶著身分權杖呼叫一次 `?mode=TEST`，檢查報表
4. `gcloud scheduler jobs create http`：每 4 小時 K 線收盤後 1 分鐘執行
5. 確認沒問題後再把模式改成 `LIMIT`

想換成自己的策略（第 8 章作業 8-A1），修改 `main.py` 裡的 `trend_strategy` 與 `TRADING_METHODS`，
先在這本筆記本試跑，再重新部署。

### 下單前請注意

- 在幣安建立 API 金鑰時，只勾選「現貨交易」，**不要**開啟提現權限，並設定 IP 白名單。
- 交易的幣種很多時請用 `LIMIT` 單：曾有人在 MAX 交易所用市價單賣出 83 枚 BTC，成交價從 15000 被砸到 6000。
- 課程程式碼是公開的，每次下單前後都請人工確認部位與訂單；程式難免有 bug，造成的損失我們無法負責。
- 部署到雲端時請固定 `finlab_crypto` 的版本（例如 `==0.3.0`），避免套件更新改變行為。